# LLMs from Scratch: train a bigger model on a free GPU

Companion notebook for the blog series [*LLMs from Scratch*](https://ilameya.github.io/blog/llm-from-scratch/).

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

The laptop model in the series has 0.8M parameters. Here we train a ~10.7M-parameter model with a longer
context (256 characters). On a free T4 GPU this takes roughly half an hour (lower `--steps` for a quicker run).

In [ ]:
!git clone -q https://github.com/ilameya/llm-from-scratch.git
%cd llm-from-scratch
!pip -q install -r requirements.txt
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (enable a GPU runtime!)")

## 1. Sanity checks
Eight quick tests: tokenizer round-trips, causality, overfitting a single batch, KV cache equivalence.

In [ ]:
!python -m pytest -q

## 2. Train
Same code as `python -m tinyllm.train` on your laptop, just bigger: 6 layers, 6 heads, width 384.
Feel free to change any of the numbers.

In [ ]:
!python -m tinyllm.train --n_layer 6 --n_head 6 --n_embd 384 --block_size 256 \
    --batch_size 64 --lr 6e-4 --min_lr 6e-5 --steps 4000 --dropout 0.2 \
    --eval_interval 500 --out checkpoints/shakespeare_big.pt

## 3. Plot the loss

In [ ]:
import json
import matplotlib.pyplot as plt
h = json.load(open("checkpoints/shakespeare_big_history.json"))
plt.figure(figsize=(7, 4))
plt.plot(h["batch_loss"], alpha=0.3, label="single batch")
plt.plot(h["step"], h["train"], "o-", label="train")
plt.plot(h["step"], h["val"], "o-", label="validation")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3)
plt.title(f"final validation loss {h['val'][-1]:.3f}")
plt.show()

## 4. Generate
Try different prompts, temperatures, `--top_k` and `--top_p` (see Part 7).

In [ ]:
!python -m tinyllm.sample --ckpt checkpoints/shakespeare_big.pt --prompt "JULIET:" --tokens 500 --temperature 0.8

## 5. Your turn
* Upload your own `.txt` file, change `load_shakespeare()` in `tinyllm/data.py` to read it, and retrain.
* Try the BPE tokenizer: `--tokenizer bpe --bpe_vocab 1024`.
* Download your checkpoint (`checkpoints/shakespeare_big.pt`) and run `python -m tinyllm.sample --ckpt ...` on your laptop.